# Create a Knowledge Graph from Text

## Task 1: Import Libraries

In [11]:
import wikipedia as wp
import re
import requests
import spacy
import spacy_transformers
from spacy import displacy
from spacy.matcher import Matcher
import networkx as nx
from pyvis.network import Network

## Task 2: Load the Data

In [12]:
wp.set_lang("en")
title = " 'Si Ronda' "
data = wp.page(title).content

print(data)

Si Ronda is a 1930 silent film from the Dutch East Indies which was directed by Lie Tek Swie and starred Bachtiar Effendi. Based on contemporary Betawi oral tradition, it follows the exploits of a bandit, skilled in silat (traditional Malay martial arts), known as Si Ronda. In the lenong stories from which the film was derived, Ronda was often depicted as a Robin Hood type of figure. The production, now thought lost, was one of a series of martial arts films released between 1929 and 1931. Si Ronda received little coverage in the media upon its release. A second adaptation of the tale, Si Ronda Macan Betawi, was made in 1978.


== Production ==
Si Ronda was adapted from a lenong (a Betawi oral tradition similar to a stage play) popular with ethnic Chinese and native audiences of the time. The Ronda stories follow the Betawi bandit of the same name, who is skilled at silat (traditional martial arts) and reputed to take from the rich to give to the poor. The Indonesian film scholar Misba

## Task 3: Preprocess the Data

In [13]:
# Convert the data to lowercase and replace new lines
data = data.lower().replace('\n', "")

# Remove the last part of the text, certain punctuation marks, headings, as well as any text within the parentheses
data = re.sub('== see also ==.*|[@#:&\"]|===.*?===|==.*?==|\(.*?\)', '', data)

# View the data
print(data)


si ronda is a 1930 silent film from the dutch east indies which was directed by lie tek swie and starred bachtiar effendi. based on contemporary betawi oral tradition, it follows the exploits of a bandit, skilled in silat , known as si ronda. in the lenong stories from which the film was derived, ronda was often depicted as a robin hood type of figure. the production, now thought lost, was one of a series of martial arts films released between 1929 and 1931. si ronda received little coverage in the media upon its release. a second adaptation of the tale, si ronda macan betawi, was made in 1978.si ronda was adapted from a lenong  popular with ethnic chinese and native audiences of the time. the ronda stories follow the betawi bandit of the same name, who is skilled at silat  and reputed to take from the rich to give to the poor. the indonesian film scholar misbach yusa biran suggests that ronda was selected for adaptation because of its potential action sequences. in the domestic cinema

## Task 4: Recognize Named Entities

In [14]:
# Load a language model
nlp = spacy.load('en_core_web_lg')
doc=nlp(data)

# Display the entities in the doc
displacy.render(doc,style="ent",jupyter=True)



## Task 5: Compute Coreference Clusters

In [15]:
# Add the coreference resolution component in the pipeline
nlp.add_pipe('coreferee')

# Pass the data to the language model 
doc = nlp(data)

# Print resolved coreferences, if any
doc._.coref_chains.print()

0: ronda(1), ronda(46), ronda(59), ronda(94)
1: film(6), film(55)
2: swie(18), it(31)
3: coverage(97), its(102)
4: ronda(121), ronda(179), its(186), ronda(214)
5: release(265), release(277)
6: pitung(332), him(338)
7: swie(357), company(377)
8: dasima(382), it(426)
9: film(403), film(438), film(492)
10: effendi(406), his(416), effendi(447)
11: ronda(441), it(450)
12: newspapers(473), it(476)
13: biran(488), he(497), his(531)
14: tan(515), tan(524), tan(536), tan(557), tan(581)
15: effendi(552), he(563)
16: film(571), film(590)
17: archives.another(631), it(656)
18: marlina(666), his(668), his(676)
19: ronda(674), ronda(688)


## Task 6: Resolve Coreferences

In [16]:
resolved_data = ""
for token in doc:
    resolved_coref = doc._.coref_chains.resolve(token)
    if resolved_coref:
        resolved_data += " " + " and ".join(r.text for r in resolved_coref)
    elif token.dep_ == "punct":
        resolved_data += token.text
    else:
        resolved_data += " " + token.text
print(resolved_data)


 si ronda is a 1930 silent film from the dutch east indies which was directed by lie tek swie and starred bachtiar effendi. based on contemporary betawi oral tradition, swie follows the exploits of a bandit, skilled in silat, known as si ronda. in the lenong stories from which the film was derived, ronda was often depicted as a robin hood type of figure. the production, now thought lost, was one of a series of martial arts films released between 1929 and 1931. si ronda received little coverage in the media upon coverage release. a second adaptation of the tale, si ronda macan betawi, was made in 1978.si ronda was adapted from a lenong   popular with ethnic chinese and native audiences of the time. the ronda stories follow the betawi bandit of the same name, who is skilled at silat   and reputed to take from the rich to give to the poor. the indonesian film scholar misbach yusa biran suggests that ronda was selected for adaptation because of ronda potential action sequences. in the dome

## Task 7: Extract Relationships

In [17]:
def extract_relationship(sentence):
    doc = nlp(sentence)
    
    first, last = None, None
    
    for chunk in doc.noun_chunks:
        if not first:
            first = chunk
        else:
            last = chunk

    if first and last:
        return (first.text.strip(), last.text.strip(), str(doc[first.end:last.start]).strip())
    
    return (None, None, None)


## Task 8: Create a Graph

In [18]:
#A helper function that prints 5 words per row. Can be used for better readability of a given text.
print_five_words = lambda sentence: '\n'.join(' '.join(sentence.split()[i:i+5]) for i in range(0, len(sentence.split()), 5))

In [21]:
# Create a Network object
graph_doc = nlp(resolved_data)

# Create an empty graph
nx_graph = nx.DiGraph()

for sent in enumerate(graph_doc.sents) :
    if len(sent[1]) > 3:
        (a, b, c) = extract_relationship(str(sent[1]))

        # Add nodes and edges to graph
        if a and b:
            nx_graph.add_node(a, size = 5)
            nx_graph.add_node(b, size = 5)
            nx_graph.add_edge(a, b, weight=1, title=print_five_words(c), arrows="to")

g = Network(notebook=True, cdn_resources='in_line')
g.from_nx(nx_graph)
g.save_graph("saved.html")
g.show("example.html")

example.html


## Task 9: List the Related Entities

In [22]:
print(nx_graph.edges(['manhattan']))

[]
